# 02 — Few-Shot and Chain-of-Thought

- **Few-shot**: provide examples directly in the prompt
- **User/assistant format**: preferred for chat models
- **How many examples**: diminishing returns after ~5
- **Self-consistency**: sample N times, take majority vote

## 0. Setup

In [ ]:
from openai import OpenAI
import json

# SET YOUR API KEY HERE
api_key = "your-api-key-here"
client = OpenAI(api_key=api_key)

print("✓ Client initialized")

In [ ]:
import os

FIXTURES = os.path.abspath(os.path.join("fixtures", "input"))
if not os.path.exists(FIXTURES):
    FIXTURES = os.path.abspath(os.path.join("..", "fixtures", "input"))

with open(os.path.join(FIXTURES, "tickets.json")) as f:
    tickets = json.load(f)

print(f"✓ Loaded {len(tickets)} tickets")
print("Example:", json.dumps(tickets[0], indent=2))

In [ ]:
def parse_json_safe(text: str) -> dict | None:
    """Parse JSON from LLM response, handling markdown code fences."""
    text = text.strip()
    if text.startswith("```"):
        lines = text.split("\n")
        text = "\n".join(lines[1:-1])
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None

print("✓ Helper defined: parse_json_safe")

## 1. Few-Shot vs Zero-Shot — Direct Comparison

In [ ]:
SYSTEM = """Classify support tickets.
Categories: billing | technical | account | shipping
Priorities: high | medium | low
Return JSON: {"category": "...", "priority": "..."}
"""

FEW_SHOT_EXAMPLES = [
    {"input": "I was charged $49.99 twice this billing cycle.",
     "output": '{"category": "billing", "priority": "high"}'},
    {"input": "App crashes every time I try to upload a file.",
     "output": '{"category": "technical", "priority": "medium"}'},
    {"input": "I cannot find the setting to update my email address.",
     "output": '{"category": "account", "priority": "low"}'},
    {"input": "My order was marked delivered but I never received it.",
     "output": '{"category": "shipping", "priority": "high"}'},
]

test_ticket = tickets[4]["text"]  # pick a ticket
print(f"Test ticket: {test_ticket[:70]}...")
print(f"Expected: {tickets[4]['category']}/{tickets[4]['priority']}\n")

# Zero-shot
r_zero = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": test_ticket}],
    temperature=0.0,
)
print(f"Zero-shot: {r_zero.choices[0].message.content}")

In [ ]:
# Few-shot via user/assistant format
def build_messages_few_shot(system, examples, query):
    messages = [{"role": "system", "content": system}]
    for ex in examples:
        messages.append({"role": "user",      "content": ex["input"]})
        messages.append({"role": "assistant", "content": ex["output"]})
    messages.append({"role": "user", "content": query})
    return messages

msgs = build_messages_few_shot(SYSTEM, FEW_SHOT_EXAMPLES, test_ticket)
r_few = client.chat.completions.create(model="gpt-4o-mini", messages=msgs, temperature=0.0)
print(f"Few-shot:  {r_few.choices[0].message.content}")

## 2. How Many Examples? — Accuracy Experiment

In [ ]:
# Compare 0, 1, 2, 4 examples on a sample of tickets
import time

sample = tickets[:8]  # use 8 tickets for speed

for n_examples in [0, 2, 4]:
    correct = 0
    for t in sample:
        examples = FEW_SHOT_EXAMPLES[:n_examples]
        if n_examples == 0:
            msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": t["text"]}]
        else:
            msgs = build_messages_few_shot(SYSTEM, examples, t["text"])
        r = client.chat.completions.create(model="gpt-4o-mini", messages=msgs, temperature=0.0)
        result = parse_json_safe(r.choices[0].message.content)
        if result and result.get("category") == t["category"]:
            correct += 1
    print(f"n_examples={n_examples}: {correct}/{len(sample)} = {correct/len(sample):.0%}")

## 3. Chain-of-Thought with Few-Shot

In [ ]:
COT_EXAMPLES = [
    {
        "input": "I was charged $49.99 twice this billing cycle.",
        "output": (
            '{"reasoning": "Core issue: duplicate charge. Category: billing (financial). '
            'Impact: direct financial loss = high.", "category": "billing", "priority": "high"}'
        ),
    },
    {
        "input": "App crashes on iOS 17 when uploading files > 10MB.",
        "output": (
            '{"reasoning": "Core issue: crash on upload. Category: technical (software bug). '
            'Has workaround (smaller files) = medium.", "category": "technical", "priority": "medium"}'
        ),
    },
]

COT_SYSTEM = """You are a ticket classifier. For each ticket:
1. Identify the core problem
2. Choose the category (billing/technical/account/shipping)
3. Assess severity/impact and set priority (high/medium/low)

Return JSON: {"reasoning": "...", "category": "...", "priority": "..."}
"""

msgs = build_messages_few_shot(COT_SYSTEM, COT_EXAMPLES, tickets[11]["text"])
r = client.chat.completions.create(model="gpt-4o-mini", messages=msgs, temperature=0.0)
result = parse_json_safe(r.choices[0].message.content)
print(json.dumps(result, indent=2))
print(f"\nExpected: {tickets[11]['category']}/{tickets[11]['priority']}")

## 4. Self-Consistency

In [ ]:
from collections import Counter

def classify_self_consistent(client, ticket_text, system, n_samples=5):
    """Sample N times at temperature>0, return majority vote."""
    votes = []
    for _ in range(n_samples):
        r = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "system", "content": system}, {"role": "user", "content": ticket_text}],
            temperature=0.7,
        )
        result = parse_json_safe(r.choices[0].message.content)
        if result and "category" in result and "priority" in result:
            votes.append((result["category"], result["priority"]))

    if not votes:
        return None
    most_common, count = Counter(votes).most_common(1)[0]
    return {"category": most_common[0], "priority": most_common[1],
            "votes": count, "total": len(votes), "confidence": count / len(votes)}

# Use on an ambiguous ticket (low priority — just question about pricing)
result = classify_self_consistent(client, tickets[4]["text"], SYSTEM, n_samples=5)
print(json.dumps(result, indent=2))